# Cycle 3 — Modelling: Player Injury Risk

**Project:** Football Predictor  
**Depends on:** `cycle3_preprocessing_injuries.ipynb`  
**Dataset:** `data/processed/player_injuries_processed.csv`

---

## Purpose

Train and evaluate baseline models for player injury risk prediction. The target is `High_Injury` (1 = player misses 28+ days this season, 0 = misses fewer than 28 days). This is a binary classification problem with a 70/30 class imbalance.

## Why AUC-ROC?

A dummy classifier that always predicts High Injury achieves 70.1% accuracy. AUC-ROC measures real discriminative ability regardless of threshold. An AUC of 0.5 = random, 1.0 = perfect.

## Injury Prediction is Hard

Injury prediction is widely acknowledged as one of the most difficult problems in sports analytics. Published models in sports science literature typically achieve AUC between 0.60 and 0.70. The fundamental challenge is that many causal injury factors (training load, pitch conditions, mental fatigue, specific tackle events) are not captured in the features available to us. The dataset provides physical attributes and historical injury data — useful, but incomplete.

---
## Cell 1 — Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, roc_curve
from xgboost import XGBClassifier

df = pd.read_csv('../../data/processed/player_injuries_processed.csv')
X = df.drop(columns=['High_Injury'])
y = df['High_Injury']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

spw = y_train.value_counts()[0] / y_train.value_counts()[1]

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'High Injury rate - Train: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%')
print(f'scale_pos_weight: {spw:.2f}')
print(f'Features ({len(X.columns)}): {list(X.columns)}')

### Output
```
Train: 1040 | Test: 261
High Injury rate - Train: 70.2% | Test: 70.1%
scale_pos_weight: 0.42
Features (17): ['height_cm','weight_kg','pace','physic','fifa_rating','age','cumulative_minutes_played',
 'cumulative_games_played','minutes_per_game_prev_seasons','avg_days_injured_prev_seasons',
 'avg_games_per_season_prev_seasons','bmi','work_rate_numeric','position_numeric',
 'significant_injury_prev_season','cumulative_days_injured','season_days_injured_prev_season']
```

### Observations
- Note: scale_pos_weight = 0.42 (less than 1.0) because the majority class is High Injury (70.2%), not Low Injury. XGBoost needs to upweight the minority class (Low Injury, 29.8%)
- The dataset is small: only 1,040 training rows. This limits complex model performance and explains why tuning gains will be modest

---
## Cell 2 — Dummy Classifier

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train_s, y_train)
y_pred_d = dummy.predict(X_test_s)
y_prob_d = dummy.predict_proba(X_test_s)[:,1]

print('DUMMY CLASSIFIER')
print(f'  Accuracy: {accuracy_score(y_test, y_pred_d)*100:.2f}%  |  AUC: {roc_auc_score(y_test, y_prob_d):.4f}')
print(classification_report(y_test, y_pred_d, target_names=['Low Injury','High Injury']))

### Output
```
Accuracy: 70.11%  |  AUC: 0.5000
Low Injury recall: 0.00 -- model predicts High Injury for everything
```

### Observations
- 70.11% accuracy from always predicting High Injury -- confirms why accuracy is misleading
- AUC 0.5 = random guessing. All real models must exceed this

---
## Cell 3 — Logistic Regression

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
y_prob_lr = lr.predict_proba(X_test_s)[:,1]

print('LOGISTIC REGRESSION')
print(f'  Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%  |  AUC: {roc_auc_score(y_test, y_prob_lr):.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['Low Injury','High Injury']))

### Output
```
Accuracy: 55.17%  |  AUC: 0.6220  <- Best baseline

              precision  recall  f1-score  support
  Low Injury       0.36    0.62      0.45       78
 High Injury       0.76    0.52      0.62      183
    accuracy                         0.55      261
```

### Observations
- AUC 0.6220 -- best baseline model despite lowest accuracy (55.17%)
- Balanced weighting causes more Low Injury predictions, trading accuracy for better discrimination
- Low Injury recall 0.62 -- correctly identifies 62% of low-risk players
- Consistent with sports science literature: linear models perform well when injury risk has an approximately linear relationship with history metrics

---
## Cell 4 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train)
y_pred_rf = rf.predict(X_test_s)
y_prob_rf = rf.predict_proba(X_test_s)[:,1]

print('RANDOM FOREST')
print(f'  Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}%  |  AUC: {roc_auc_score(y_test, y_prob_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['Low Injury','High Injury']))

### Output
```
Accuracy: 72.03%  |  AUC: 0.5916

 Low Injury recall: 0.14 -- almost never predicts Low Injury correctly
High Injury recall: 0.97
```

### Observations
- High accuracy (72.03%) but AUC only 0.5916 -- near-random discrimination
- Model is essentially predicting High Injury for almost everyone (recall 0.97)
- This shows Random Forest with default settings collapses to majority-class prediction despite balanced weighting

---
## Cell 5 — XGBoost

In [ ]:
xgb = XGBClassifier(scale_pos_weight=spw, random_state=42, eval_metric='auc', verbosity=0)
xgb.fit(X_train_s, y_train)
y_pred_xgb = xgb.predict(X_test_s)
y_prob_xgb = xgb.predict_proba(X_test_s)[:,1]

print('XGBOOST')
print(f'  Accuracy: {accuracy_score(y_test, y_pred_xgb)*100:.2f}%  |  AUC: {roc_auc_score(y_test, y_prob_xgb):.4f}')
print(classification_report(y_test, y_pred_xgb, target_names=['Low Injury','High Injury']))

### Output
```
Accuracy: 65.52%  |  AUC: 0.6179

  Low Injury  precision=0.41  recall=0.36
 High Injury  precision=0.74  recall=0.78
```

### Observations
- AUC 0.6179 -- competitive with LR. scale_pos_weight correctly pushes XGBoost to predict Low Injury more
- Better than RF at discrimination (0.6179 vs 0.5916) but worse than LR (0.6220)
- Will be tuned in next notebook

---
## Cell 6 — ROC Curve Comparison

In [ ]:
import os
os.makedirs('../../docs', exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 6))
for name, probs, color in [
    ('Dummy',               y_prob_d,   'gray'),
    ('Logistic Regression', y_prob_lr,  'steelblue'),
    ('Random Forest',       y_prob_rf,  'seagreen'),
    ('XGBoost',             y_prob_xgb, 'tomato'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Cycle 3 -- ROC Curves: Injury Risk Baseline Models')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../../docs/cycle3_roc_baseline.png', dpi=150)
plt.show()

---
## Cell 7 — Full Baseline Summary

In [ ]:
results = pd.DataFrame([
    {'Model': 'Dummy',               'Accuracy': 70.11, 'AUC-ROC': 0.5000},
    {'Model': 'Logistic Regression', 'Accuracy': 55.17, 'AUC-ROC': 0.6220},
    {'Model': 'Random Forest',       'Accuracy': 72.03, 'AUC-ROC': 0.5916},
    {'Model': 'XGBoost',             'Accuracy': 65.52, 'AUC-ROC': 0.6179},
])
print(results.sort_values('AUC-ROC', ascending=False).to_string(index=False))
print()
print('Best baseline: Logistic Regression (AUC=0.6220)')
print('Note: AUC 0.60-0.70 is the expected range for injury prediction.')
print('Published sports science models typically report AUC 0.60-0.70.')

### Output
```
                Model  Accuracy  AUC-ROC
  Logistic Regression     55.17   0.6220  <- Best
              XGBoost     65.52   0.6179
        Random Forest     72.03   0.5916
                Dummy     70.11   0.5000
```

### Key Observations
- All models above dummy -- the features provide genuine (if limited) signal
- LR leads untuned -- injury risk has an approximately linear relationship with history metrics
- RF high accuracy = majority-class collapse (AUC near random despite 72% accuracy)
- The tight clustering around 0.60-0.62 shows the fundamental difficulty of injury prediction

---
**Next:** `cycle3_tuning.ipynb` -- tune XGBoost and RF, compare final results